# Build a 3LC table from `rfcx/frugalai`

Uses the HF audio bytes fastpath added to `_TableFromHuggingFaceBase`:
raw WAV bytes from parquet are written verbatim to disk under the table's
`bulk_data/`, and the table itself stores only string URLs.

The `wav_audio` sample type (from the sibling `audio-sample-type` example,
extended in this POC to accept `EncodedSample`) is resolved via the
`tlc.sample_types` entry point — make sure that package is `pip install -e`'d
before running this notebook.

In [1]:
import datasets
import tlc

### Build the train and val tables

We materialize 10k rows from each split (`train` for training, `test` for
validation). One `.wav` file per row gets written under the table's
`bulk_data/` — bytes-verbatim, no decode/re-encode round trip.

10k rows ≈ 700 MB on disk per table.

In [2]:
N_ROWS = 10_000

PROJECT = "audio-classification-poc"
DATASET = "frugalai"


def build_table(split: str, table_name: str):
    hf_ds = datasets.load_dataset("rfcx/frugalai", split=split).select(range(N_ROWS))
    return tlc.Table.from_hugging_face_dataset(
        hf_ds,
        project_name=PROJECT,
        dataset_name=DATASET,
        table_name=table_name,
        if_exists="overwrite",
    )


train_table = build_table(split="train", table_name=f"train-{N_ROWS}")
val_table = build_table(split="test", table_name=f"val-{N_ROWS}")

print(f"train: {len(train_table)} rows  url={train_table.url}")
print(f"val:   {len(val_table)} rows  url={val_table.url}")

# The remaining cells inspect `table` — alias it to the val table so the
# rest of the notebook still illustrates row-view / sample-view / verify.
table = val_table

Loading and flattening samples from frugalai:  10%|#         | 1.02k/10.0k [00:00<00:04, 1.90kit/s]

Loading and flattening samples from frugalai:  23%|##3       | 2.31k/10.0k [00:00<00:01, 4.37kit/s]

train: 10000 rows  url=/Users/gudbrand/Library/Application Support/3LC/projects/audio-classification-poc/datasets/frugalai/tables/train-10000
val:   10000 rows  url=/Users/gudbrand/Library/Application Support/3LC/projects/audio-classification-poc/datasets/frugalai/tables/val-10000


### Row view: just URL strings

The schema/parquet representation of the audio column is a plain string
(role `URL/Audio`). The Dashboard and the parquet file see only paths.

In [3]:
print("row 0:", dict(table.table_rows[0]))
print("row 1:", dict(table.table_rows[1]))

row 0: {'audio': '../../bulk_data/samples/val-10000/audio/0000000.wav', 'label': 1, 'weight': 1.0}
row 1: {'audio': '../../bulk_data/samples/val-10000/audio/0000001.wav', 'label': 0, 'weight': 1.0}


In [ ]:
audio_schema = table.row_schema['audio']
print(audio_schema)

{
  "sample_type":"wav_audio",
  "value":{
    "type":"string",
    "string_role":"URL/Audio"
  }
}


### Sample view: NumPy waveforms

On read, the `wav_audio` sample type loads the WAV file into a `float32`
1-D array. This is what the PyTorch Dataset will consume in the next
notebook.

In [5]:
sample = table[0]
print(f"audio: ndarray, shape={sample['audio'].shape}, dtype={sample['audio'].dtype}")
print(f"label: {sample['label']}")

audio: ndarray, shape=(36000,), dtype=float32
label: 1


### Sanity check: bytes copied verbatim

Compare the bytes in the materialized `.wav` file against the bytes that
HF parquet hands us with `decode=False`. They should be byte-identical —
no decode/re-encode happened during table creation.

In [6]:
from pathlib import Path

# Re-load the val split with decode=False so we can compare raw parquet bytes
# against the on-disk WAV the table materialized.
hf_raw = (
    datasets.load_dataset("rfcx/frugalai", split="test")
    .select(range(N_ROWS))
    .cast_column("audio", datasets.Audio(decode=False))
)
hf_bytes = hf_raw[0]['audio']['bytes']

wav_url = tlc.Url(table.table_rows[0]['audio']).to_absolute(table.url)
wav_bytes = Path(wav_url.to_str()).read_bytes()

print(f"hf parquet bytes: {len(hf_bytes)} ({hf_bytes[:4]!r})")
print(f"on-disk wav bytes: {len(wav_bytes)} ({wav_bytes[:4]!r})")
print(f"identical: {hf_bytes == wav_bytes}")

hf parquet bytes: 72044 (b'RIFF')
on-disk wav bytes: 72044 (b'RIFF')
identical: True
